In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# 🔹 1. LOAD DATA
# -------------------------------
df = pd.read_csv("online_retail.csv")

# -------------------------------
# 🔹 2. DATA CLEANING
# -------------------------------
# Remove missing customer IDs
df = df.dropna(subset=['Customer ID'])

# Remove negative or zero quantity
df = df[df['Quantity'] > 0]

# Convert Customer ID to integer
df['Customer ID'] = df['Customer ID'].astype(int)

# -------------------------------
# 🔹 3. CREATE USER-ITEM MATRIX
# -------------------------------
user_item = df.pivot_table(
    index='Customer ID',
    columns='StockCode',
    values='Quantity',
    aggfunc='sum'
).fillna(0)

# -------------------------------
# 🔹 4. USER SIMILARITY
# -------------------------------
user_similarity = cosine_similarity(user_item)

user_sim_df = pd.DataFrame(
    user_similarity,
    index=user_item.index,
    columns=user_item.index
)

# -------------------------------
# 🔹 5. RECOMMENDATION FUNCTION
# -------------------------------
def recommend_products(user_id, top_n=5):
    
    if user_id not in user_item.index:
        return "User not found!"
    
    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:10]
    
    weighted_scores = np.zeros(user_item.shape[1])
    
    for sim_user, score in similar_users.items():
        weighted_scores += score * user_item.loc[sim_user].values
    
    recommendations = pd.Series(weighted_scores, index=user_item.columns)
    
    # Remove already purchased products
    purchased = user_item.loc[user_id]
    recommendations = recommendations[purchased == 0]
    
    return recommendations.sort_values(ascending=False).head(top_n)

# -------------------------------
# 🔹 6. GET PRODUCT NAMES
# -------------------------------
products = df[['StockCode', 'Description']].drop_duplicates()

def get_product_names(product_ids):
    return products[products['StockCode'].isin(product_ids)]

# -------------------------------
# 🔹 7. TEST
# -------------------------------
user_id = user_item.index[0]  # first available user

rec = recommend_products(user_id)

print("Recommended Product IDs:")
print(rec)

print("\nRecommended Product Names:")
print(get_product_names(rec.index))

Recommended Product IDs:
StockCode
47566    84.460878
23298    70.125248
23230    58.437707
22938    57.028790
23167    56.792970
dtype: float64

Recommended Product Names:
       StockCode                     Description
1329       47566                   PARTY BUNTING
346817     22938        CUPCAKE LACE PAPER SET 6
581415     23230            WRAP ALPHABET DESIGN
700158     23298                  SPOTTY BUNTING
702683     23298               BUNTING , SPOTTY 
711380     23167  SMALL CERAMIC TOP STORAGE JAR 
